In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
import lightgbm as lgb
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier


import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers
import gc
import os

from sklearn.linear_model import LogisticRegression
from scipy.stats import ks_2samp, spearmanr

import matplotlib.pyplot as plt

from sklearn.isotonic import IsotonicRegression
import joblib
import json

In [ ]:
train = pd.read_csv("/kaggle/input/diabetes-prediction/train.csv")
test = pd.read_csv("/kaggle/input/diabetes-prediction/test.csv")

train.columns

In [ ]:
SEED = 42
TARGET = "diagnosed_diabetes"

train["kfold"] = -1

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train[TARGET])):
    train.loc[val_idx, "kfold"] = fold

train["kfold"].value_counts()

In [ ]:
cat_cols = ['gender', 'ethnicity', 'education_level', 'income_level', 
            'smoking_status', 'employment_status', 'family_history_diabetes', 
            'hypertension_history', 'cardiovascular_history']

os.makedirs("models/lgbm", exist_ok=True)
os.makedirs("models/cb", exist_ok=True)
os.makedirs("models/et", exist_ok=True)


features = [c for c in train.columns if c not in ['id', 'kfold', TARGET]]
num_cols = [c for c in features if c not in cat_cols]

# Label Encoding 
for c in cat_cols:
    le = LabelEncoder()
    train[c] = le.fit_transform(train[c].astype(str))
    test[c] = test[c].map(lambda s: le.transform([str(s)])[0] if str(s) in le.classes_ else -1)

# GLOBAL BINNING
train_bin = train.copy()
test_bin = test.copy()

for c in num_cols:
    # Learn bins from TRAIN
    _, bins = pd.qcut(train[c], q=15, retbins=True, duplicates='drop')
    
    # Extend edges to infinity to handle outliers in Test
    bins[0] = -np.inf
    bins[-1] = np.inf
    
    # Apply bins to Train and Test
    train_bin[c] = pd.cut(train[c], bins=bins, labels=False)
    test_bin[c] = pd.cut(test[c], bins=bins, labels=False)
    
    # Fill potential NaNs (rare with inf edges) with bin like -1
    train_bin[c] = train_bin[c].fillna(-1).astype(int)
    test_bin[c] = test_bin[c].fillna(-1).astype(int)

oof_lgbm = np.zeros(len(train))
oof_cb = np.zeros(len(train))
oof_et = np.zeros(len(train))

pred_lgbm = np.zeros(len(test))
pred_cb = np.zeros(len(test))
pred_et = np.zeros(len(test))

# Training Loop
print(f"Starting Training on {len(features)} features...")
for fold in range(10):
    print("Fold: ", fold)
    val_idx = train['kfold'] == fold
    train_idx = train['kfold'] != fold
    
    # Standard Data
    X_tr, y_tr = train.loc[train_idx, features], train.loc[train_idx, TARGET]
    X_val, y_val = train.loc[val_idx, features], train.loc[val_idx, TARGET]
    
    # Binned Data (For CatBoost)
    X_tr_bin, X_val_bin = train_bin.loc[train_idx, features], train_bin.loc[val_idx, features]

    # Model 1: LightGBM
    clf_lgbm = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=7,
        num_leaves=32,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        device_type='gpu',
        gpu_platform_id=0,
        gpu_device_id=0,
        metric='auc',
        verbosity=-1
    )

    clf_lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
                 callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False),
                            lgb.log_evaluation(period=100)])

    oof_lgbm[val_idx] = clf_lgbm.predict_proba(X_val)[:, 1]
    pred_lgbm += clf_lgbm.predict_proba(test[features])[:, 1] / 10
    best_iter = clf_lgbm.best_iteration_
    clf_lgbm.booster_.save_model(f"models/lgbm/lgbm_fold_{fold}.txt", num_iteration=best_iter)


    # Model 2: CatBoost
    clf_cb = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.03,
        depth=6,
        eval_metric='AUC',
        random_seed=SEED,
        verbose=0,
        allow_writing_files=False,
        task_type='GPU',
        devices='0'
    )
    clf_cb.fit(X_tr_bin, y_tr, eval_set=(X_val_bin, y_val), early_stopping_rounds=100)
    oof_cb[val_idx] = clf_cb.predict_proba(X_val_bin)[:, 1]
    pred_cb += clf_cb.predict_proba(test_bin[features])[:, 1] / 10
    clf_cb.save_model(f"models/cb/cb_fold_{fold}.cbm")


    # Model 3: ExtraTrees
    clf_et = ExtraTreesClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=75,
        max_features='sqrt',
        random_state=SEED,
        n_jobs=-1
    )
    clf_et.fit(X_tr, y_tr)
    oof_et[val_idx] = clf_et.predict_proba(X_val)[:, 1]
    pred_et += clf_et.predict_proba(test[features])[:, 1] / 10
    import joblib
    joblib.dump(clf_et, f"models/et/et_fold_{fold}.pkl")


# Scores
auc_lgbm = roc_auc_score(train[TARGET], oof_lgbm)
auc_cb = roc_auc_score(train[TARGET], oof_cb)
auc_et = roc_auc_score(train[TARGET], oof_et)

print(f"LGBM AUC: {auc_lgbm:.5f}")
print(f"CatBoost (Binned) AUC: {auc_cb:.5f}")
print(f"ExtraTrees AUC: {auc_et:.5f}")

In [ ]:
os.makedirs("models", exist_ok=True)
strategy = tf.distribute.MirroredStrategy()
oof_nn = np.zeros(len(train))
pred_nn = np.zeros(len(test))

print("Starting Neural Network Training...")

for fold in range(10):
    val_idx = train['kfold'] == fold
    train_idx = train['kfold'] != fold
    
    # Data Split
    X_tr_raw = train.loc[train_idx, features]
    X_val_raw = train.loc[val_idx, features]
    X_test_raw = test[features]
    
    y_tr = train.loc[train_idx, TARGET].values
    y_val = train.loc[val_idx, TARGET].values
    
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr_raw)
    X_val = scaler.transform(X_val_raw)
    X_test_fold = scaler.transform(X_test_raw)
    
    # Clear Memory
    tf.keras.backend.clear_session()
    gc.collect()
    
    with strategy.scope():
        model = models.Sequential([
            layers.Input(shape=(len(features),)),
            
            layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
            layers.BatchNormalization(),
            layers.Dropout(0.3),
            
            layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
            layers.BatchNormalization(),
            layers.Dropout(0.3),
            
            layers.Dense(1, activation='sigmoid')
        ])
        
        model.compile(
            optimizer='adam', 
            loss='binary_crossentropy', 
            metrics=[tf.keras.metrics.AUC(name='auc')]
        )
    
    # Callbacks
    es = callbacks.EarlyStopping(monitor='val_auc', patience=10, mode='max', restore_best_weights=True)
    lr = callbacks.ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=5, verbose=0)
    
    # Train
    model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=512,
        callbacks=[es, lr],
        verbose=0
    )
    
    # Predict
    oof_nn[val_idx] = model.predict(X_val, batch_size=512, verbose=0).ravel()
    pred_nn += model.predict(X_test_fold, batch_size=512, verbose=0).ravel() / 10
    
    # Save Model
    model.save(f"models/nn_fold_{fold}.keras")
    print(f"Fold {fold} Complete.")

# Score
auc_nn = roc_auc_score(train[TARGET], oof_nn)
print(f"Neural Network AUC: {auc_nn:.5f}")

In [ ]:

os.makedirs("models", exist_ok=True)

importances = []

for fold in range(10):
    path = f"/kaggle/working/models/lgbm/lgbm_fold_{fold}.txt"
    if os.path.exists(path):
        booster = lgb.Booster(model_file=path)
        importances.append(
            booster.feature_importance(importance_type="gain")
        )

if len(importances) == 10:
    avg_importance = np.mean(importances, axis=0)
    feature_imp = pd.Series(avg_importance, index=features)
    top_15_features = feature_imp.sort_values(ascending=False).head(15).index.tolist()
else:
    print("Falling back to correlation-based features")
    top_15_features = (
        train[features]
        .corrwith(train[TARGET])
        .abs()
        .sort_values(ascending=False)
        .head(15)
        .index.tolist()
    )


oof_lr_15 = np.zeros(len(train))
pred_lr_15 = np.zeros(len(test))

oof_lr_full = np.zeros(len(train))
pred_lr_full = np.zeros(len(test))

for fold in range(10):
    val_idx = train['kfold'] == fold
    train_idx = train['kfold'] != fold
    
    X_tr_15_raw = train.loc[train_idx, top_15_features]
    X_val_15_raw = train.loc[val_idx, top_15_features]
    X_test_15_raw = test[top_15_features]
    
    scaler_15 = StandardScaler()
    X_tr_15 = scaler_15.fit_transform(X_tr_15_raw)
    X_val_15 = scaler_15.transform(X_val_15_raw)
    X_test_15 = scaler_15.transform(X_test_15_raw)
    
    clf_lr_15 = LogisticRegression(penalty='l2', C=1.0, random_state=SEED, solver='lbfgs', max_iter=1000)
    clf_lr_15.fit(X_tr_15, train.loc[train_idx, TARGET])
    
    oof_lr_15[val_idx] = clf_lr_15.predict_proba(X_val_15)[:, 1]
    pred_lr_15 += clf_lr_15.predict_proba(X_test_15)[:, 1] / 10
    
    X_tr_full_raw = train.loc[train_idx, features]
    X_val_full_raw = train.loc[val_idx, features]
    X_test_full_raw = test[features]
    
    scaler_full = StandardScaler()
    X_tr_full = scaler_full.fit_transform(X_tr_full_raw)
    X_val_full = scaler_full.transform(X_val_full_raw)
    X_test_full = scaler_full.transform(X_test_full_raw)
    
    clf_lr_full = LogisticRegression(penalty='l2', C=0.1, random_state=SEED, solver='lbfgs', max_iter=1000)
    clf_lr_full.fit(X_tr_full, train.loc[train_idx, TARGET])
    
    oof_lr_full[val_idx] = clf_lr_full.predict_proba(X_val_full)[:, 1]
    pred_lr_full += clf_lr_full.predict_proba(X_test_full)[:, 1] / 10

    joblib.dump(clf_lr_15, f"models/lr_top15_fold_{fold}.pkl")
    joblib.dump(clf_lr_full, f"models/lr_full_fold_{fold}.pkl")
    joblib.dump(scaler_15, f"models/scaler_lr_top15_fold_{fold}.pkl")
    joblib.dump(scaler_full, f"models/scaler_lr_full_fold_{fold}.pkl")

auc_lr_15 = roc_auc_score(train[TARGET], oof_lr_15)
auc_lr_full = roc_auc_score(train[TARGET], oof_lr_full)

ks_stat, _ = ks_2samp(oof_lr_15, oof_lr_full)

print(f"LR Top-15 AUC: {auc_lr_15:.5f}")
print(f"LR Full AUC:   {auc_lr_full:.5f}")
print(f"KS Statistic:  {ks_stat:.5f}")

In [ ]:

# 1. Aggregate OOF Predictions
oof_df = pd.DataFrame({
    'LGBM': oof_lgbm,
    'CatBoost': oof_cb,
    'ExtraTrees': oof_et,
    'NN': oof_nn,
    'LR_Top15': oof_lr_15,
    'Target': train[TARGET]
})

# 2. Identify Best Model
auc_scores = {col: roc_auc_score(oof_df['Target'], oof_df[col]) for col in oof_df.columns if col != 'Target'}
best_model_name = max(auc_scores, key=auc_scores.get)
print(f"Best Model (Anchor): {best_model_name} (AUC: {auc_scores[best_model_name]:.5f})")

# 3. Statistical Gate & Visual Test
selected_models = [best_model_name]
plt.figure(figsize=(10, 6))

# Plot Anchor CDF
sorted_anchor = np.sort(oof_df[best_model_name])
yvals = np.arange(len(sorted_anchor)) / float(len(sorted_anchor) - 1)
plt.plot(sorted_anchor, yvals, label=f"{best_model_name} (Anchor)", linewidth=2, color='black')

print("\n--- Diversity Audit ---")
for model in oof_df.columns:
    if model in ['Target', best_model_name]:
        continue

    # Stats
    ks_stat, _ = ks_2samp(oof_df[best_model_name], oof_df[model])
    corr, _ = spearmanr(oof_df[best_model_name], oof_df[model])
    
    # Gate Logic
    is_diverse = (ks_stat > 0.015) or (corr < 0.985)
    status = "PASS" if is_diverse else "FAIL"
    
    if is_diverse:
        selected_models.append(model)
        
    print(f"{model.ljust(12)} | KS: {ks_stat:.4f} | Corr: {corr:.4f} | {status}")

    # Plot CDF
    sorted_model = np.sort(oof_df[model])
    plt.plot(sorted_model, yvals, label=f"{model} (KS={ks_stat:.3f})", alpha=0.7)

plt.title("CDF Separation Test (Visual Diversity)")
plt.xlabel("Prediction Probability")
plt.ylabel("Cumulative Frequency")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nFinal Ensemble Candidates: {selected_models}")

In [ ]:
joblib.dump(selected_models, "models/selected_models_list.pkl")

In [ ]:
selected_models = joblib.load("models/selected_models_list.pkl")

pred_df = pd.DataFrame({
    'LGBM': pred_lgbm,
    'CatBoost': pred_cb,
    'ExtraTrees': pred_et,
    'NN': pred_nn,
    'LR_Top15': pred_lr_15
})

oof_rank = oof_df[selected_models].rank(pct=True)
pred_rank = pred_df[selected_models].rank(pct=True)

current_best_auc = 0
weights = {model: 0 for model in selected_models}
ensemble_oof = np.zeros(len(train))
ensemble_pred = np.zeros(len(test))

iterations = 100

for i in range(iterations):
    best_model = None
    best_auc = current_best_auc
    
    for model in selected_models:
        potential_weight = (weights[model] + 1) / (i + 1)
        if potential_weight > 0.50:
            continue
            
        temp_ensemble = (ensemble_oof + oof_rank[model]) / (i + 1)
        temp_auc = roc_auc_score(train[TARGET], temp_ensemble)
        
        if temp_auc >= best_auc:
            best_auc = temp_auc
            best_model = model
            
    if best_model is not None:
        weights[best_model] += 1
        ensemble_oof += oof_rank[best_model]
        ensemble_pred += pred_rank[best_model]
        current_best_auc = best_auc

total_votes = sum(weights.values())
assert total_votes > 0, "Ensemble has zero weight — check constraints"

final_weights = {m: w/total_votes for m, w in weights.items()}
ensemble_oof /= total_votes
ensemble_pred /= total_votes

iso_reg = IsotonicRegression(out_of_bounds='clip')
iso_reg.fit(ensemble_oof, train[TARGET])

final_predictions = iso_reg.transform(ensemble_pred)
calibrated_oof = iso_reg.transform(ensemble_oof)
final_auc = roc_auc_score(train[TARGET], calibrated_oof)

print(f"Weights: {json.dumps(final_weights, indent=2)}")
print(f"Final AUC: {final_auc:.5f}")

submission = pd.DataFrame({
    'id': test['id'],
    TARGET: final_predictions
})
submission.to_csv("submission.csv", index=False)

In [ ]:
joblib.dump(final_weights, "models/ensemble_weights.pkl")
joblib.dump(iso_reg, "models/isotonic_calibrator.pkl")
joblib.dump(final_predictions, "models/final_test_preds.pkl")

In [ ]:
print("Weight sum:", sum(final_weights.values()))
print("Max single weight:", max(final_weights.values()))


In [ ]:
print("Pre-calibration AUC:", roc_auc_score(train[TARGET], ensemble_oof))
print("Post-calibration AUC:", final_auc)


In [ ]:


os.makedirs("models", exist_ok=True)

manual_weights = {
    'LGBM': 0.40,
    'NN': 0.20,
    'CatBoost': 0.15,
    'LR_Top15': 0.15,
    'ExtraTrees': 0.10
}

assert abs(sum(manual_weights.values()) - 1.0) < 1e-6, "Weights must sum to 1.0"


oof_df = pd.DataFrame({
    'LGBM': oof_lgbm,
    'NN': oof_nn,
    'CatBoost': oof_cb,
    'LR_Top15': oof_lr_15,
    'ExtraTrees': oof_et,
    'Target': train[TARGET]
})

pred_df = pd.DataFrame({
    'LGBM': pred_lgbm,
    'NN': pred_nn,
    'CatBoost': pred_cb,
    'LR_Top15': pred_lr_15,
    'ExtraTrees': pred_et
})


oof_rank = oof_df.drop(columns='Target').rank(pct=True)
pred_rank = pred_df.rank(pct=True)


ensemble_oof = np.zeros(len(train))
ensemble_pred = np.zeros(len(test))

for model, w in manual_weights.items():
    ensemble_oof += w * oof_rank[model].values
    ensemble_pred += w * pred_rank[model].values

ensemble_oof /= sum(manual_weights.values())
ensemble_pred /= sum(manual_weights.values())


pre_auc = roc_auc_score(train[TARGET], ensemble_oof)
print(f"Pre-calibration AUC: {pre_auc:.6f}")


iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(ensemble_oof, train[TARGET])

calibrated_oof = iso.transform(ensemble_oof)
calibrated_pred = iso.transform(ensemble_pred)

post_auc = roc_auc_score(train[TARGET], calibrated_oof)
print(f"Post-calibration AUC: {post_auc:.6f}")
print(f"Delta: {post_auc - pre_auc:+.6f}")


joblib.dump(manual_weights, "models/final_ensemble_weights.pkl")
joblib.dump(iso, "models/isotonic_calibrator.pkl")
joblib.dump(calibrated_pred, "models/final_test_predictions.pkl")

with open("models/final_ensemble_weights.json", "w") as f:
    json.dump(manual_weights, f, indent=2)


submission = pd.DataFrame({
    'id': test['id'],
    TARGET: calibrated_pred
})

submission.to_csv("submission_random.csv", index=False)

print("\nFINAL SUBMISSION SAVED: submission.csv")
print("ENSEMBLE COMPLETE")
